# Day 5 — Decorators, Context Managers & Caching

## Objective
Demonstrate cross-cutting concern primitives: custom `@timeit` decorator, parameterized `@retry` decorator with `functools.wraps`, resource context managers (`TaskResourceManager` & `@contextmanager`), and performance optimization via `functools.lru_cache`.

## 1. Environment & Imports

In [1]:
import sys
import time
from pathlib import Path
repo_root = Path.cwd().resolve()
sys.path.insert(0, str(repo_root / 'src'))

from task_analytics import (
    timeit, retry, TaskResourceManager, managed_resource, calculate_task_priority_score, get_task_category_weight, Pipeline, CleanDataStep
)
print('Day 5 decorators, context managers, and caching modules loaded.')

Day 5 decorators, context managers, and caching modules loaded.


## 2. Custom `@timeit` Decorator & `functools.wraps`

In [2]:
@timeit
def process_tasks_mock(tasks):
    """Mock function processing tasks with artificial delay."""
    time.sleep(0.02)
    return len(tasks)

print('Function metadata preserved by wraps:')
print('  __name__:', process_tasks_mock.__name__)
print('  __doc__: ', process_tasks_mock.__doc__)
count = process_tasks_mock([{'id': 1}, {'id': 2}])
print('Result:', count)

Function metadata preserved by wraps:
  __name__: process_tasks_mock
  __doc__:  Mock function processing tasks with artificial delay.
Result: 2


## 3. Parameterized `@retry(max_attempts=N)` Decorator

Demonstrate successful retry recovery and max retry exhaustion.

In [3]:
attempt_count = 0
@retry(max_attempts=3)
def flaky_network_call():
    global attempt_count
    attempt_count += 1
    print(f'  Attempt {attempt_count} executing...')
    if attempt_count < 2:
        raise ConnectionError('Temporary glitch')
    return {'status': 'success'}

print('--- 1. Controlled Retry Recovery ---')
res = flaky_network_call()
print('Success result:', res)

print('\n--- 2. Exhausting Max Retries ---')
fail_attempts = 0
@retry(max_attempts=3)
def always_failing_call():
    global fail_attempts
    fail_attempts += 1
    print(f'  Failing attempt {fail_attempts}...')
    raise TimeoutError('Service unavailable')

try:
    always_failing_call()
except TimeoutError as e:
    print('Caught final exception after 3 retries:', e)

--- 1. Controlled Retry Recovery ---
  Attempt 1 executing...
  Attempt 2 executing...
Success result: {'status': 'success'}

--- 2. Exhausting Max Retries ---
  Failing attempt 1...
  Failing attempt 2...
  Failing attempt 3...
Caught final exception after 3 retries: Service unavailable


Attempt 1/3 for 'flaky_network_call' failed: Temporary glitch. Retrying...
Attempt 1/3 for 'always_failing_call' failed: Service unavailable. Retrying...
Attempt 2/3 for 'always_failing_call' failed: Service unavailable. Retrying...
Attempt 3/3 for 'always_failing_call' failed: Service unavailable. All retries exhausted.


## 4. Context Managers & Resource Safety

Demonstrate resource cleanup under normal execution and exception flows.

In [4]:
print('--- 1. Class Context Manager Normal Flow ---')
with TaskResourceManager('prod_db') as res:
    print('  Inside with block:', res.resource_data)
print('  Outside block: is_open =', res.is_open)

print('\n--- 2. Class Context Manager Exception Flow ---')
try:
    with TaskResourceManager('temp_buffer') as res:
        raise RuntimeError('Disk full inside block!')
except RuntimeError as e:
    print('  Caught exception outside block:', e)
    print('  Verified Cleanup: is_open =', res.is_open)

--- 1. Class Context Manager Normal Flow ---
  Inside with block: {'status': 'active', 'name': 'prod_db'}
  Outside block: is_open = False

--- 2. Class Context Manager Exception Flow ---
  Caught exception outside block: Disk full inside block!
  Verified Cleanup: is_open = False


[Resource Manager] Cleaning up resource 'temp_buffer' after exception: Disk full inside block!


## 5. `functools.lru_cache` & Caching Dangers

Demonstrate caching performance gains, cache clearing, and immutable return safety.

In [5]:
calculate_task_priority_score.cache_clear()
t0 = time.perf_counter()
s1 = calculate_task_priority_score('high', 10, 2)
t1 = time.perf_counter()

t2 = time.perf_counter()
s2 = calculate_task_priority_score('high', 10, 2)
t3 = time.perf_counter()

print(f'First call (computed):  {(t1-t0)*1e6:.2f} µs')
print(f'Second call (cache hit): {(t3-t2)*1e6:.2f} µs')
print('Cache Info:', calculate_task_priority_score.cache_info())
calculate_task_priority_score.cache_clear()
print('Cache Stats after cache_clear():', calculate_task_priority_score.cache_info())

First call (computed):  18.40 µs
Second call (cache hit): 1.40 µs
Cache Info: CacheInfo(hits=1, misses=1, maxsize=128, currsize=1)
Cache Stats after cache_clear(): CacheInfo(hits=0, misses=0, maxsize=128, currsize=0)


## Conclusion & Key Takeaways

- **`functools.wraps`**: Essential for preserving function signature and metadata when wrapping functions.
- **Cleanup Guarantee**: `with` blocks guarantee cleanup even when exceptions occur.
- **Caching Caution**: Always specify `maxsize` to prevent unbounded memory growth, and return immutable types to prevent state corruption.